In [1]:
# periksa checkpoint epoch 10
import os
import torch

output_dir = r"C:\Faster-RCNN-PPE\experiments\fasterrcnn_20_epochs"

checkpoint_path = os.path.join(
    output_dir,
    "fasterrcnn_epoch_10.pth"
)

print("=== PEMERIKSAAN CHECKPOINT EPOCH 10 ===")
print("Checkpoint ditemukan:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    print("Epoch tersimpan       :", checkpoint["epoch"])
    print("Average epoch loss    :", checkpoint["average_epoch_loss"])
    print("Total epoch loss      :", checkpoint["total_epoch_loss"])
    print("Jumlah kelas          :", checkpoint["num_classes"])
    print("Ukuran file           :", round(
        os.path.getsize(checkpoint_path) / (1024**2), 2
    ), "MB")

=== PEMERIKSAAN CHECKPOINT EPOCH 10 ===
Checkpoint ditemukan: True
Epoch tersimpan       : 10
Average epoch loss    : 0.19837502819071606
Total epoch loss      : 235.8679085187614
Jumlah kelas          : 9
Ukuran file           : 315.03 MB


In [2]:
# Menyiapkan model untuk evaluasi
model.eval()

print("=== PERSIAPAN EVALUASI FASTER R-CNN ===")
print("Mode model       :", "Evaluation" if not model.training else "Training")
print("Device           :", device)
print("Jumlah data val  :", len(val_dataset))
print("Jumlah batch val :", len(val_loader))

NameError: name 'model' is not defined

In [3]:
# Periksa sesi jupyter
print("=== PEMERIKSAAN SESI JUPYTER ===")

for nama in ["device", "val_dataset", "val_loader", "model"]:
    print(f"{nama:15}:", nama in globals())

=== PEMERIKSAAN SESI JUPYTER ===
device         : False
val_dataset    : False
val_loader     : False
model          : False


In [4]:
# Import library
import os
import torch
import torchvision
import torch.nn.functional as F
import torch_directml

import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from types import MethodType

print("Import library berhasil.")
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

Import library berhasil.
PyTorch: 2.4.1+cpu
Torchvision: 0.19.1+cpu


In [5]:
# Inisialisasi perangkat directML
device = torch_directml.device()

print("=== PEMERIKSAAN DEVICE ===")
print("Device       :", device)
print("PyTorch      :", torch.__version__)
print("Torchvision  :", torchvision.__version__)

=== PEMERIKSAAN DEVICE ===
Device       : privateuseone:0
PyTorch      : 2.4.1+cpu
Torchvision  : 0.19.1+cpu


In [6]:
# Menentukan lokasi dataset
dataset_root = r"C:\Faster-RCNN-PPE\dataset\images"

val_image_dir = os.path.join(dataset_root, "val")

print("=== PEMERIKSAAN DATASET VALIDASI ===")
print("Dataset root :", dataset_root)
print("Folder val   :", val_image_dir)
print("Folder ditemukan:", os.path.exists(val_image_dir))

if os.path.exists(val_image_dir):
    print("Jumlah file JPG:", len([
        f for f in os.listdir(val_image_dir)
        if f.lower().endswith((".jpg", ".jpeg"))
    ]))
    print("Jumlah file XML:", len([
        f for f in os.listdir(val_image_dir)
        if f.lower().endswith(".xml")
    ]))

=== PEMERIKSAAN DATASET VALIDASI ===
Dataset root : C:\Faster-RCNN-PPE\dataset\images
Folder val   : C:\Faster-RCNN-PPE\dataset\images\val
Folder ditemukan: True
Jumlah file JPG: 170
Jumlah file XML: 170


In [7]:
# Pemetaan Kelas
class_names = {
    1: "person",
    2: "vest",
    3: "yellow",
    4: "white",
    5: "glass",
    6: "blue",
    7: "red",
    8: "head"
}

num_classes = len(class_names) + 1

print("=== PEMETAAN KELAS ===")
print("Jumlah kelas termasuk background:", num_classes)
print("0: background")

for class_id, class_name in class_names.items():
    print(f"{class_id}: {class_name}")

=== PEMETAAN KELAS ===
Jumlah kelas termasuk background: 9
0: background
1: person
2: vest
3: yellow
4: white
5: glass
6: blue
7: red
8: head


In [8]:
# Membuat dataset class untuk validasi
class CHVGPpeDataset(Dataset):
    def __init__(self, image_dir, class_names):
        self.image_dir = image_dir
        self.class_names = class_names

        self.image_files = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg"))
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_name = self.image_files[idx]

        image_path = os.path.join(self.image_dir, image_name)
        xml_path = os.path.join(
            self.image_dir,
            os.path.splitext(image_name)[0] + ".xml"
        )

        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        image_tensor = torchvision.transforms.functional.to_tensor(image)

        tree = ET.parse(xml_path)
        root = tree.getroot()

        boxes = []
        labels = []

        for obj in root.findall("object"):
            class_name = obj.find("name").text.strip()

            if class_name not in self.class_names.values():
                continue

            class_id = next(
                key for key, value in self.class_names.items()
                if value == class_name
            )

            bbox = obj.find("bndbox")

            xmin = max(0, min(float(bbox.find("xmin").text), width))
            ymin = max(0, min(float(bbox.find("ymin").text), height))
            xmax = max(0, min(float(bbox.find("xmax").text), width))
            ymax = max(0, min(float(bbox.find("ymax").text), height))

            if xmax <= xmin or ymax <= ymin:
                continue

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(class_id)

        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (
                (boxes[:, 2] - boxes[:, 0]) *
                (boxes[:, 3] - boxes[:, 1])
            ),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64)
        }

        return image_tensor, target

In [9]:
# Membuat dataset validasi
val_dataset = CHVGPpeDataset(
    image_dir=val_image_dir,
    class_names=class_names
)

print("=== DATASET VALIDASI ===")
print("Jumlah gambar validasi:", len(val_dataset))

=== DATASET VALIDASI ===
Jumlah gambar validasi: 170


In [10]:
sample_image, sample_target = val_dataset[0]

print("=== PEMERIKSAAN SAMPEL DATA VALIDASI ===")
print("Ukuran tensor gambar :", sample_image.shape)
print("Ukuran bounding box  :", sample_target["boxes"].shape)
print("Jumlah objek         :", len(sample_target["labels"]))
print("ID kelas             :", sample_target["labels"].tolist())
print("Bounding box         :", sample_target["boxes"].tolist())

=== PEMERIKSAAN SAMPEL DATA VALIDASI ===
Ukuran tensor gambar : torch.Size([3, 640, 640])
Ukuran bounding box  : torch.Size([9, 4])
Jumlah objek         : 9
ID kelas             : [8, 8, 8, 8, 8, 1, 1, 1, 1]
Bounding box         : [[18.0, 102.0, 151.0, 212.0], [377.0, 126.0, 439.0, 203.0], [310.0, 123.0, 393.0, 212.0], [181.0, 158.0, 239.0, 201.0], [263.0, 150.0, 322.0, 210.0], [166.0, 163.0, 257.0, 421.0], [380.0, 132.0, 508.0, 534.0], [3.0, 103.0, 152.0, 631.0], [267.0, 132.0, 393.0, 636.0]]


In [11]:
# Membuat DataLoader Validasi
def collate_fn(batch):
    return tuple(zip(*batch))


val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

print("=== DATALOADER VALIDASI ===")
print("Jumlah gambar :", len(val_dataset))
print("Batch size    :", val_loader.batch_size)
print("Jumlah batch  :", len(val_loader))
print("Shuffle       : False")
print("Workers       : 0")

=== DATALOADER VALIDASI ===
Jumlah gambar : 170
Batch size    : 1
Jumlah batch  : 170
Shuffle       : False
Workers       : 0


In [12]:
# Membangun arsitektur Faster R-CNN
# Memuat arsitektur Faster R-CNN dengan pretrained COCO weights
model = fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
    min_size=640,
    max_size=640
)

# Mengganti predictor agar sesuai dengan 9 kelas
in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes
)

print("=== ARSITEKTUR FASTER R-CNN ===")
print("Model              : Faster R-CNN ResNet50 FPN")
print("Ukuran input       : 640 x 640")
print("Jumlah kelas       :", num_classes)
print("Predictor features:", in_features)

=== ARSITEKTUR FASTER R-CNN ===
Model              : Faster R-CNN ResNet50 FPN
Ukuran input       : 640 x 640
Jumlah kelas       : 9
Predictor features: 1024


In [13]:
# Memasang 4 patch directML
from torchvision.models.detection import roi_heads as roi_heads_module

# Patch 1: RPN compute_loss
def compute_loss_safe(self, objectness, pred_bbox_deltas,
                      labels, regression_targets):

    sampled_pos_inds, sampled_neg_inds = self.fg_bg_sampler(labels)

    sampled_pos_inds = torch.where(
        torch.cat(sampled_pos_inds, dim=0)
    )[0]

    sampled_neg_inds = torch.where(
        torch.cat(sampled_neg_inds, dim=0)
    )[0]

    sampled_inds = torch.cat([
        sampled_pos_inds,
        sampled_neg_inds
    ])

    objectness = objectness.reshape(-1)
    labels = torch.cat(labels, dim=0).reshape(-1)
    pred_bbox_deltas = pred_bbox_deltas.reshape(-1, 4)
    regression_targets = torch.cat(
        regression_targets, dim=0
    ).reshape(-1, 4)

    if sampled_inds.numel() > 0:
        objectness_loss = F.binary_cross_entropy_with_logits(
            objectness[sampled_inds],
            labels[sampled_inds].to(objectness.dtype)
        )
    else:
        objectness_loss = objectness.sum() * 0.0

    if sampled_pos_inds.numel() > 0:
        box_loss = F.smooth_l1_loss(
            pred_bbox_deltas[sampled_pos_inds],
            regression_targets[sampled_pos_inds],
            beta=1/9,
            reduction="sum"
        ) / max(sampled_inds.numel(), 1)
    else:
        box_loss = pred_bbox_deltas.sum() * 0.0

    return objectness_loss, box_loss


model.rpn.compute_loss = MethodType(
    compute_loss_safe, model.rpn
)


# Patch 2: model forward
original_forward = model.forward

def forward_safe(self, images, targets=None):
    if self.training and targets is not None:
        for target in targets:
            boxes = target["boxes"]

            if boxes.numel() > 0:
                if not torch.isfinite(boxes).all():
                    raise ValueError("Bounding box mengandung nilai non-finite.")

                if (boxes[:, 2:] <= boxes[:, :2]).any():
                    raise ValueError("Bounding box memiliki ukuran tidak valid.")

    return original_forward(images, targets)


model.forward = MethodType(forward_safe, model)


# Patch 3: ROI add_gt_proposals
original_add_gt_proposals = model.roi_heads.add_gt_proposals

def add_gt_proposals_safe(self, proposals, gt_boxes):
    return original_add_gt_proposals(proposals, gt_boxes)


model.roi_heads.add_gt_proposals = MethodType(
    add_gt_proposals_safe, model.roi_heads
)


# Patch 4: Fast R-CNN loss
def fastrcnn_loss_safe(class_logits, box_regression, labels, regression_targets):

    labels = torch.cat(labels, dim=0)
    regression_targets = torch.cat(regression_targets, dim=0)

    classification_loss = F.cross_entropy(
        class_logits, labels
    )

    sampled_pos_inds = torch.where(labels > 0)[0]
    labels_pos = labels[sampled_pos_inds]

    N, num_classes = class_logits.shape

    box_regression = box_regression.reshape(N, -1, 4)

    if sampled_pos_inds.numel() > 0:
        box_loss = F.smooth_l1_loss(
            box_regression[sampled_pos_inds, labels_pos],
            regression_targets[sampled_pos_inds],
            beta=1/9,
            reduction="sum"
        )
        box_loss = box_loss / max(labels.numel(), 1)
    else:
        box_loss = box_regression.sum() * 0.0

    return classification_loss, box_loss


roi_heads_module.fastrcnn_loss = fastrcnn_loss_safe


print("=== PEMASANGAN PATCH DIRECTML ===")
print("Patch 1:", model.rpn.compute_loss.__func__.__name__)
print("Patch 2:", model.forward.__func__.__name__)
print("Patch 3:", model.roi_heads.add_gt_proposals.__func__.__name__)
print("Patch 4:", roi_heads_module.fastrcnn_loss.__name__)

=== PEMASANGAN PATCH DIRECTML ===
Patch 1: compute_loss_safe
Patch 2: forward_safe
Patch 3: add_gt_proposals_safe
Patch 4: fastrcnn_loss_safe


In [14]:
# Memuat checkpoint epoch 10
# Lokasi checkpoint
checkpoint_path = r"C:\Faster-RCNN-PPE\experiments\fasterrcnn_20_epochs\fasterrcnn_epoch_10.pth"

# Memastikan checkpoint tersedia
assert os.path.exists(checkpoint_path), "Checkpoint tidak ditemukan!"

# Memuat checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Memuat bobot hasil training
model.load_state_dict(checkpoint["model_state_dict"])

# Memindahkan model ke DirectML
model.to(device)

print("=== CHECKPOINT BERHASIL DIMUAT ===")
print("Epoch training :", checkpoint["epoch"])
print("Average loss   :", checkpoint["average_epoch_loss"])
print("Total loss     :", checkpoint["total_epoch_loss"])
print("Jumlah kelas   :", checkpoint["num_classes"])
print("Device         :", device)

=== CHECKPOINT BERHASIL DIMUAT ===
Epoch training : 10
Average loss   : 0.19837502819071606
Total loss     : 235.8679085187614
Jumlah kelas   : 9
Device         : privateuseone:0


In [15]:
model.eval()

print("=== VERIFIKASI MODEL EVALUASI ===")
print("Model training epoch :", checkpoint["epoch"])
print("Mode model           :", "Evaluation" if not model.training else "Training")
print("Device               :", device)
print("Jumlah kelas         :", num_classes)
print("Jumlah data validasi :", len(val_dataset))
print("Jumlah batch         :", len(val_loader))

=== VERIFIKASI MODEL EVALUASI ===
Model training epoch : 10
Mode model           : Evaluation
Device               : privateuseone:0
Jumlah kelas         : 9
Jumlah data validasi : 170
Jumlah batch         : 170


In [16]:
# Inferensi data validasi
import time

all_predictions = []
all_targets = []

model.eval()

start_time = time.time()

with torch.no_grad():
    for images, targets in val_loader:

        images = [image.to(device) for image in images]

        outputs = model(images)

        for output in outputs:
            prediction = {
                "boxes": output["boxes"].detach().cpu(),
                "labels": output["labels"].detach().cpu(),
                "scores": output["scores"].detach().cpu()
            }
            all_predictions.append(prediction)

        for target in targets:
            ground_truth = {
                "boxes": target["boxes"].cpu(),
                "labels": target["labels"].cpu()
            }
            all_targets.append(ground_truth)

        if len(all_predictions) % 10 == 0:
            print(
                f"Proses inferensi: "
                f"{len(all_predictions)}/{len(val_dataset)} gambar"
            )

end_time = time.time()

print("\n=== INFERENSI SELESAI ===")
print("Jumlah prediksi gambar :", len(all_predictions))
print("Jumlah ground truth    :", len(all_targets))
print("Waktu inferensi        :", round(end_time - start_time, 2), "detik")

Proses inferensi: 10/170 gambar
Proses inferensi: 20/170 gambar
Proses inferensi: 30/170 gambar
Proses inferensi: 40/170 gambar
Proses inferensi: 50/170 gambar
Proses inferensi: 60/170 gambar
Proses inferensi: 70/170 gambar
Proses inferensi: 80/170 gambar
Proses inferensi: 90/170 gambar
Proses inferensi: 100/170 gambar
Proses inferensi: 110/170 gambar
Proses inferensi: 120/170 gambar
Proses inferensi: 130/170 gambar
Proses inferensi: 140/170 gambar
Proses inferensi: 150/170 gambar
Proses inferensi: 160/170 gambar
Proses inferensi: 170/170 gambar

=== INFERENSI SELESAI ===
Jumlah prediksi gambar : 170
Jumlah ground truth    : 170
Waktu inferensi        : 137.81 detik


In [18]:
# Memeriksa hasil prediksi
print("=== PEMERIKSAAN HASIL PREDIKSI ===")

total_predictions = sum(
    len(prediction["boxes"])
    for prediction in all_predictions
)

total_ground_truth = sum(
    len(target["boxes"])
    for target in all_targets
)

print("Total prediksi model :", total_predictions)
print("Total ground truth   :", total_ground_truth)

print("\n=== CONTOH PREDIKSI GAMBAR PERTAMA ===")

first_prediction = all_predictions[0]

print("Jumlah deteksi :", len(first_prediction["boxes"]))
print("Label kelas    :", first_prediction["labels"].tolist())
print("Confidence     :", first_prediction["scores"].tolist())

=== PEMERIKSAAN HASIL PREDIKSI ===
Total prediksi model : 1679
Total ground truth   : 1211

=== CONTOH PREDIKSI GAMBAR PERTAMA ===
Jumlah deteksi : 13
Label kelas    : [1, 1, 8, 1, 1, 8, 8, 8, 8, 2, 8, 8, 8]
Confidence     : [0.9990777969360352, 0.9985548853874207, 0.9971845149993896, 0.9963709115982056, 0.9952747225761414, 0.9942328333854675, 0.9887363314628601, 0.9810470938682556, 0.970832347869873, 0.2053488940000534, 0.19615168869495392, 0.1275257170200348, 0.055139362812042236]


In [21]:
# Menghitung TP, FP, FN
# IoU : 0.5
# Confidence : 0.4
from torchvision.ops import box_iou

CONF_THRESHOLD = 0.40
IOU_THRESHOLD = 0.50

TP = 0
FP = 0
FN = 0

for prediction, target in zip(all_predictions, all_targets):

    pred_boxes = prediction["boxes"]
    pred_labels = prediction["labels"]
    pred_scores = prediction["scores"]

    gt_boxes = target["boxes"]
    gt_labels = target["labels"]

    # Filter prediksi berdasarkan confidence
    keep = pred_scores >= CONF_THRESHOLD

    pred_boxes = pred_boxes[keep]
    pred_labels = pred_labels[keep]
    pred_scores = pred_scores[keep]

    # Urutkan prediksi dari confidence tertinggi
    order = torch.argsort(pred_scores, descending=True)

    pred_boxes = pred_boxes[order]
    pred_labels = pred_labels[order]

    matched_gt = set()

    for i in range(len(pred_boxes)):

        pred_box = pred_boxes[i]
        pred_label = pred_labels[i]

        # Ground truth yang belum dipasangkan dan kelasnya sama
        candidates = [
            j for j in range(len(gt_boxes))
            if j not in matched_gt and gt_labels[j] == pred_label
        ]

        if not candidates:
            FP += 1
            continue

        candidate_boxes = gt_boxes[candidates]

        ious = box_iou(
            pred_box.unsqueeze(0),
            candidate_boxes
        )[0]

        best_idx = torch.argmax(ious).item()
        best_iou = ious[best_idx].item()

        if best_iou >= IOU_THRESHOLD:
            TP += 1
            matched_gt.add(candidates[best_idx])
        else:
            FP += 1

    FN += len(gt_boxes) - len(matched_gt)


print("=== HASIL PENCocOKAN PREDIKSI ===")
print("Confidence threshold :", CONF_THRESHOLD)
print("IoU threshold        :", IOU_THRESHOLD)
print("TP                   :", TP)
print("FP                   :", FP)
print("FN                   :", FN)

=== HASIL PENCocOKAN PREDIKSI ===
Confidence threshold : 0.4
IoU threshold        : 0.5
TP                   : 1081
FP                   : 183
FN                   : 130


In [22]:
# Precision, Recall, F1-Score
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

f1_score = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0 else 0
)

print("=== HASIL EVALUASI FASTER R-CNN EPOCH 10 ===")
print(f"Precision : {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall    : {recall:.4f} ({recall * 100:.2f}%)")
print(f"F1-score  : {f1_score:.4f} ({f1_score * 100:.2f}%)")

=== HASIL EVALUASI FASTER R-CNN EPOCH 10 ===
Precision : 0.8552 (85.52%)
Recall    : 0.8927 (89.27%)
F1-score  : 0.8735 (87.35%)


In [23]:
# Periksa dataset test
test_image_dir = os.path.join(dataset_root, "test")

print("=== PEMERIKSAAN DATASET TEST ===")
print("Folder test ditemukan:", os.path.exists(test_image_dir))

if os.path.exists(test_image_dir):
    test_images = [
        f for f in os.listdir(test_image_dir)
        if f.lower().endswith((".jpg", ".jpeg"))
    ]

    test_xml = [
        f for f in os.listdir(test_image_dir)
        if f.lower().endswith(".xml")
    ]

    print("Jumlah gambar JPG:", len(test_images))
    print("Jumlah file XML:", len(test_xml))

=== PEMERIKSAAN DATASET TEST ===
Folder test ditemukan: True
Jumlah gambar JPG: 340
Jumlah file XML: 340


In [24]:
# Membuat test_dataset
test_dataset = CHVGPpeDataset(
    image_dir=test_image_dir,
    class_names=class_names
)

print("Jumlah data test:", len(test_dataset))

Jumlah data test: 340


In [25]:
# Membuat test_louder
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

print("Jumlah batch test:", len(test_loader))

Jumlah batch test: 340


In [26]:
print("=== PEMERIKSAAN MODEL ===")
print("Model tersedia:", model is not None)
print("Mode evaluasi:", not model.training)
print("Device:", next(model.parameters()).device)
print("Jumlah kelas:", model.roi_heads.box_predictor.cls_score.out_features)
print("Checkpoint:", checkpoint_path)
print("Epoch checkpoint:", checkpoint["epoch"])

=== PEMERIKSAAN MODEL ===
Model tersedia: True
Mode evaluasi: True
Device: privateuseone:0
Jumlah kelas: 9
Checkpoint: C:\Faster-RCNN-PPE\experiments\fasterrcnn_20_epochs\fasterrcnn_epoch_10.pth
Epoch checkpoint: 10


In [27]:
# Siapkan variabel evaluasi
confidence_threshold = 0.40
iou_threshold = 0.50

TP = 0
FP = 0
FN = 0

print("Confidence threshold:", confidence_threshold)
print("IoU threshold:", iou_threshold)
print("TP, FP, FN berhasil diinisialisasi.")

Confidence threshold: 0.4
IoU threshold: 0.5
TP, FP, FN berhasil diinisialisasi.


In [28]:
def evaluate_faster_rcnn(model, test_loader, device,
                         confidence_threshold=0.40,
                         iou_threshold=0.50):

    TP, FP, FN = 0, 0, 0
    model.eval()

    start_time = time.time()

    with torch.no_grad():
        for index, (images, targets) in enumerate(test_loader):

            images = [image.to(device) for image in images]
            outputs = model(images)

            for output, target in zip(outputs, targets):

                # Ambil prediksi yang memenuhi confidence threshold
                scores = output["scores"].detach().cpu()
                selected = scores >= confidence_threshold

                pred_boxes = output["boxes"].detach().cpu()[selected]
                pred_labels = output["labels"].detach().cpu()[selected]

                gt_boxes = target["boxes"].cpu()
                gt_labels = target["labels"].cpu()

                matched_gt = set()

                # Urutkan prediksi berdasarkan confidence tertinggi
                order = torch.argsort(scores[selected], descending=True)

                for pred_idx in order.tolist():
                    best_iou = 0
                    best_gt_idx = -1

                    for gt_idx in range(len(gt_boxes)):
                        if gt_idx in matched_gt:
                            continue

                        if pred_labels[pred_idx].item() != gt_labels[gt_idx].item():
                            continue

                        iou = calculate_iou(
                            pred_boxes[pred_idx].tolist(),
                            gt_boxes[gt_idx].tolist()
                        )

                        if iou > best_iou:
                            best_iou = iou
                            best_gt_idx = gt_idx

                    if best_gt_idx != -1 and best_iou >= iou_threshold:
                        TP += 1
                        matched_gt.add(best_gt_idx)
                    else:
                        FP += 1

                FN += len(gt_boxes) - len(matched_gt)

            if (index + 1) % 20 == 0:
                print(f"Memproses gambar {index + 1}/{len(test_loader)}")

    precision = TP / (TP + FP) if TP + FP > 0 else 0
    recall = TP / (TP + FN) if TP + FN > 0 else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0
    )

    elapsed = time.time() - start_time

    return {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Waktu (detik)": elapsed
    }

In [29]:
# Uji di data test
test_results = evaluate_faster_rcnn(
    model=model,
    test_loader=test_loader,
    device=device,
    confidence_threshold=0.40,
    iou_threshold=0.50
)

print("\n===== HASIL PENGUJIAN FASTER R-CNN =====")
print("Jumlah gambar test :", len(test_dataset))
print("TP                 :", test_results["TP"])
print("FP                 :", test_results["FP"])
print("FN                 :", test_results["FN"])
print("Precision          : {:.4f} ({:.2f}%)".format(
    test_results["Precision"], test_results["Precision"] * 100))
print("Recall             : {:.4f} ({:.2f}%)".format(
    test_results["Recall"], test_results["Recall"] * 100))
print("F1-score           : {:.4f} ({:.2f}%)".format(
    test_results["F1-score"], test_results["F1-score"] * 100))
print("Waktu pengujian    : {:.2f} detik".format(
    test_results["Waktu (detik)"]))

NameError: name 'calculate_iou' is not defined

In [30]:
# Definisikan fungsi calculate_iou
def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box3[3]) if False else min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0

In [31]:
# Uji di data test
test_results = evaluate_faster_rcnn(
    model=model,
    test_loader=test_loader,
    device=device,
    confidence_threshold=0.40,
    iou_threshold=0.50
)

print("\n===== HASIL PENGUJIAN FASTER R-CNN =====")
print("Jumlah gambar test :", len(test_dataset))
print("TP                 :", test_results["TP"])
print("FP                 :", test_results["FP"])
print("FN                 :", test_results["FN"])
print("Precision          : {:.4f} ({:.2f}%)".format(
    test_results["Precision"], test_results["Precision"] * 100))
print("Recall             : {:.4f} ({:.2f}%)".format(
    test_results["Recall"], test_results["Recall"] * 100))
print("F1-score           : {:.4f} ({:.2f}%)".format(
    test_results["F1-score"], test_results["F1-score"] * 100))
print("Waktu pengujian    : {:.2f} detik".format(
    test_results["Waktu (detik)"]))

Memproses gambar 20/340
Memproses gambar 40/340
Memproses gambar 60/340
Memproses gambar 80/340
Memproses gambar 100/340
Memproses gambar 120/340
Memproses gambar 140/340
Memproses gambar 160/340
Memproses gambar 180/340
Memproses gambar 200/340
Memproses gambar 220/340
Memproses gambar 240/340
Memproses gambar 260/340
Memproses gambar 280/340
Memproses gambar 300/340
Memproses gambar 320/340
Memproses gambar 340/340

===== HASIL PENGUJIAN FASTER R-CNN =====
Jumlah gambar test : 340
TP                 : 2128
FP                 : 342
FN                 : 257
Precision          : 0.8615 (86.15%)
Recall             : 0.8922 (89.22%)
F1-score           : 0.8766 (87.66%)
Waktu pengujian    : 260.90 detik
